# Pipelines

In this lab we work with numerical and categorical features together in a single processing pipeline.

1. Imputing missing values
2. Scaling numerical features
3. Encoding categorical features


## 1. Load the dataset


We need to define the data and the target. Here we build a regression model.


In [ ]:
from sklearn.datasets import fetch_openml

ames_housing = fetch_openml(name="house_prices", as_frame=True)
data = ames_housing.data
target = ames_housing.target

Display the first rows of the dataframe.


In [ ]:
data.head()

## 2. Select a subset of features


**For simplicity**, we select a subset of features and work with that:


In [ ]:
numeric_features = ["LotArea", "FullBath", "HalfBath"]
categorical_features = ["Neighborhood", "HouseStyle"]
data = data[numeric_features + categorical_features]

## 3. Pre-processing

Raw data is rarely ready for use in ML models. We often need steps such as:

- Handling missing values.
- Encoding categorical variables.
- Scaling numerical variables.

To keep things simple, we choose a fixed set of columns to work with:


### a. Handling missing values (Imputation)

Missing values are common. Most algorithms do not accept gaps in the data.

**Imputation strategies (`SimpleImputer`):**

1. **`mean`:** For numeric data that is roughly normally distributed.
2. **`median`:** Better for numeric data with outliers.
3. **`most_frequent`:** Often used for categorical data.
4. **`constant`:** Fill with a fixed value (e.g. "Unknown" or 0).

**How do we put this in a pipeline?**
With a `Pipeline`, the imputer is fit only on training data, and the learned rules are then applied to the test set. This avoids data leakage.


#### 3.1 Numerical features


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

#### 3.2 Categorical features


In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

## 4. Assigning transforms to columns

Three things to remember:

1. **ColumnTransformer** [link](https://scikit-learn.org/stable/modules/compose.html#column-transformer) applies different transforms to selected feature subsets.
2. **FeatureUnion** [link](https://scikit-learn.org/stable/modules/compose.html#feature-union) combines transformer outputs into a single feature space.
3. To transform the target (e.g. log-transform of y) use **TransformedTargetRegressor** [link](https://scikit-learn.org/stable/modules/compose.html#transformed-target-regressor).


In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 5. Define the pipeline


Define the regression model.

We often call the whole pipeline the **model**. So we name the final predictor accordingly.


In [ ]:
from sklearn.linear_model import SGDRegressor

predictor = SGDRegressor(loss="squared_error")

Chain the preprocessing steps with the predictor.


In [ ]:
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", predictor),
    ]
)
pipe

### Aside: an alternative way to define a pipeline: `make_pipeline`


**Alternative:** You can use the functional API: `make_column_transformer` and `make_pipeline`. These do not require (or allow) manual names for the estimators; names are auto-generated from the class names (lowercase).


```python
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = make_column_transformer(
    (numeric_transformer, numeric_features),
    (categorical_transformer, categorical_features),
)
pipe = make_pipeline(preprocessor, SGDRegressor())
```


## Measuring generalization

A central idea in machine learning is **generalization**: how well the model performs on new data it was not trained on.

A common mistake is to evaluate the model on the same data used for training. That gives overly optimistic, misleading results.


In [ ]:
# WRONG
# pipe.fit(data, target)

### Train and test sets

Split the data into two parts:

1. **Training set:** Used to fit the model.
2. **Test set:** Held out completely; used only at the end to evaluate performance.


In [ ]:
from sklearn.model_selection import train_test_split

# Split data: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    data, target, test_size=0.2, random_state=42
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")


Now we fit the pipeline; note how the diagram updates after fitting.


In [ ]:
pipe.fit(X_train, y_train)

## 6. Evaluate performance on the test set


In [ ]:
score = pipe.score(X_test, y_test)
print(f"Score on the test set: {score:.3f}")

::: {.callout-note}

**Model evaluation** deserves its own discussion; we include it here just for completeness.

:::

---

- Source: [scikit-learn developer API design](https://scikit-learn.org/stable/developers/develop.html)
- See also: [Predictive machine learning pipeline with mixed data types](https://scikit-learn.org/stable/auto_examples/compose/plot_column_transformer_mixed_types.html).
